# Phase 3b — YOLOv8n + Coordinate Attention (Colab, T4)

Trains **YOLOv8n with one Coordinate Attention block** inserted at the end of the
backbone (Hou et al., CVPR 2021), on the same SOURCE split (India + Japan).

> **This is the with/without ablation for the novel component (the project spec 3).**
> Every hyperparameter, the seed (42), and the splits are **identical to the
> Phase 3a plain YOLOv8n run** — batch 16, 100 epochs, 640px, optimizer auto,
> patience 20. The CA block is the *only* difference. Changing anything else
> invalidates the comparison.

CA adds ~6.7k parameters (3.01 M -> ~3.02 M). `train_yolo.py --attention`
registers the module in the ultralytics parser, builds from
`config/yolov8n_ca.yaml`, transfers stock yolov8n.pt weights into the matching
layers, and logs the run under phase `3b` as `yolov8n_ca_source`.

Code comes from the small **`code_patch.zip`** you upload in Step 3b — same
file used for the YOLO26n run (it already contains `src/models/attention.py`,
`src/models/yolo_attention.py`, and `config/yolov8n_ca.yaml`). Rebuild it if
your local code changed since.

## Step 1 — Confirm a GPU is attached

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No CUDA GPU. Runtime -> Change runtime type -> GPU, then rerun."
)
print("device:", torch.cuda.get_device_name(0))

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Step 3a — Extract the DATA bundle (clean extract every time)

Edit `BUNDLE_ZIP` if your zip is not at `My Drive/rdd_bundle.zip`.

In [ ]:
import zipfile, os, shutil, time
from pathlib import Path

BUNDLE_ZIP = '/content/drive/MyDrive/rdd_bundle.zip'
REPO = Path('/content/road-damage-detection')

assert Path(BUNDLE_ZIP).exists(), f"Not found: {BUNDLE_ZIP} -- check the path in your Drive"
if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir(parents=True, exist_ok=True)

t0 = time.time()
with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    zf.extractall(REPO)
print(f"Extracted data bundle in {time.time() - t0:.0f}s")
os.chdir(REPO)
print("cwd:", os.getcwd())

## Step 3b — Upload the CODE patch and overwrite src/ + config/

Run this cell, then pick **`code_patch.zip`** from your local repo. The grep at
the end must show both the `--attention` flag and the CoordinateAttention class.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()   # select dist/code_patch.zip
patch_name = next(iter(uploaded))
with zipfile.ZipFile(patch_name) as zf:
    zf.extractall('/content/road-damage-detection')
print("patched:", patch_name)
print()
!grep -n "attention" src/models/train_yolo.py | head -3
!grep -n "class CoordinateAttention" src/models/attention.py
!test -f config/yolov8n_ca.yaml && echo "config/yolov8n_ca.yaml: present" || echo "MISSING config/yolov8n_ca.yaml"

## Step 4 — Install dependencies

In [ ]:
!pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)

## Step 5 — Point the dataset configs at Colab paths

Must run AFTER Step 3a (the extract restores Windows-path YAMLs).

In [ ]:
!python src/data/write_dataset_configs.py --data-root /content/road-damage-detection/data/processed
print()
!cat config/dataset_source.yaml

## Step 6 — Verify data + the CA module builds

Counts must match `data/split_report.json` (12,748 / 2,732 / 2,732). The second
block builds the CA model once and checks the param delta over stock yolov8n.

In [ ]:
from pathlib import Path

expected = {'train': 12748, 'val': 2732, 'test': 2732}
for split, n in expected.items():
    got = len(list(Path(f'data/processed/source/images/{split}').glob('*.jpg')))
    assert got == n, f"{split}: expected {n} images, found {got} -- upload incomplete"
print("Data counts match split_report.json.")

In [ ]:
import sys
sys.path.insert(0, 'src/models')
from attention import register_ca_module
from yolo_attention import build_ca_model

register_ca_module()
# build_ca_model(..., validate=True) does its OWN internal check: it builds
# a reference plain yolov8n with nc FORCED to match (4, not COCO's 80 -- an
# 80-class head has far more params than a 4-class one, which would swamp
# CA's +6,680 and make any naive bare-YOLO('yolov8n.pt') comparison here
# meaningless). The 'CA validation OK: ... +6,680 params' line below IS the
# real check -- nothing further to assert on top of it.
ca = build_ca_model(pretrained_weights='yolov8n.pt', validate=True)
nca = sum(p.numel() for p in ca.model.parameters())
print(f"\nyolov8n + CA total params: {nca:,}")

## Step 7 — Define helpers AND restore previous progress

> Run in EVERY session before any training chunk. Defines `save_results()` /
> `progress()` and pulls checkpoints back from Drive so a recycled VM resumes
> instead of restarting from epoch 0.

> **Run this at the START of a session, not defensively after training.**
> `restore_results()` is per-run epoch-aware (never overwrites a local run
> directory with fewer completed epochs than Drive has), so it is now safe
> to call at any point -- but the whole reason to run it early is that ANY
> training you do before calling it exists ONLY on this VM's local disk,
> unreachable until `save_results()` (defined by this same cell) actually
> runs. Restoring late doesn't destroy that local progress any more, but it
> also can't protect progress the VM itself loses if the runtime recycles
> first.

In [ ]:
import csv, shutil, os
from pathlib import Path

def _epochs_done(run_dir: Path) -> int:
    """Epochs completed for a run directory, from results.csv row count
    (same source of truth train_yolo.py itself uses) -- 0 for anything
    without one, including a directory that doesn't exist."""
    rc = run_dir / 'results.csv'
    if not rc.exists():
        return 0
    with open(rc, newline='') as f:
        return max(0, sum(1 for _ in csv.reader(f)) - 1)

def restore_results():
    """Pull Drive's saved progress in, per RUN DIRECTORY -- never wholesale.
    A run already further along locally (e.g. a chunk just finished in THIS
    session, not yet saved to Drive) is left alone rather than being
    overwritten by Drive's older copy. This is what makes it safe to call
    even after training has already produced local-only progress."""
    src_root = Path('/content/drive/MyDrive/rdd_results')
    if not src_root.exists():
        print("No saved results in Drive yet -- fresh start."); return
    runs_src = src_root / 'runs'
    if runs_src.exists():
        for run_src in sorted(runs_src.iterdir()):
            if not run_src.is_dir():
                continue
            run_dst = Path('experiments/runs') / run_src.name
            drive_epochs, local_epochs = _epochs_done(run_src), _epochs_done(run_dst)
            if drive_epochs > local_epochs:
                run_dst.parent.mkdir(parents=True, exist_ok=True)
                if run_dst.exists():
                    shutil.rmtree(run_dst)
                shutil.copytree(run_src, run_dst)
                print(f"  restored {run_src.name}: {drive_epochs} epochs (Drive) > {local_epochs} (local)")
            else:
                print(f"  kept local {run_src.name}: {local_epochs} epochs >= Drive at {drive_epochs}")
    # results/ (the CSV logs) are small and append-only -- copy in only
    # files that don't already exist locally, never overwrite one that does.
    results_src = src_root / 'results'
    if results_src.exists():
        results_dst = Path('experiments/results')
        results_dst.mkdir(parents=True, exist_ok=True)
        for f in results_src.iterdir():
            if f.is_file() and not (results_dst / f.name).exists():
                shutil.copy2(f, results_dst / f.name)
                print(f"  restored experiments/results/{f.name} (was missing locally)")

def save_results():
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        out = drive_root / 'rdd_results'
        out.mkdir(parents=True, exist_ok=True)
        for src in [Path('experiments/runs'), Path('experiments/results')]:
            if src.exists():
                dst = out / src.name
                if dst.exists():
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
        print("saved to Drive:", out)
        return out
    print("Drive not mounted -- packing for download."); 
    archive = shutil.make_archive('/content/rdd_results', 'zip', 'experiments')
    from google.colab import files
    files.download(archive)
    return Path(archive)

def progress():
    import csv as _csv, yaml as _yaml
    for run in sorted(Path('experiments/runs').glob('*')):
        rc, ay = run / 'results.csv', run / 'args.yaml'
        if rc.exists():
            done = max(0, sum(1 for _ in _csv.reader(open(rc))) - 1)
            total = (_yaml.safe_load(open(ay)) or {}).get('epochs', '?') if ay.exists() else '?'
            print(f"  {run.name}: {done}/{total} epochs")

restore_results()
print("\nhelpers ready. current progress:")
progress()

## Step 8 — Smoke test (recommended, ~1 min)

Logged as phase `3b-smoke` — a pipeline artifact, never a reportable result.

In [ ]:
!python src/models/train_yolo.py --attention \
    --data config/dataset_source_subset.yaml \
    --name yolov8n_ca_smoke \
    --epochs 2 --smoke --device 0

## Step 9 — Train YOLOv8n + CA: 100 epochs, resumable chunks

**Identical config to Phase 3a's plain yolov8n_source** — batch 16, 100 epochs,
640px, seed 42. `--chunk-epochs 20` = 5 chunks (~30-45 min each on a T4); lower
it if cells run long. **Re-run the same cell** until it reports complete — each
rerun resumes exactly where the last stopped (optimizer / EMA / LR schedule
preserved).

> New session? Re-run Steps 1-7 first (GPU, mount, extract, code patch, install,
> configs, helpers+restore). Jumping straight here in a fresh kernel gives
> `NameError: save_results is not defined`, and a recycled VM would restart from
> epoch 0.

In [ ]:
!python src/models/train_yolo.py --attention \
    --data config/dataset_source.yaml \
    --name yolov8n_ca_source \
    --chunk-epochs 20 \
    --device 0

save_results()   # persist this chunk to Drive before anything can drop

### Progress check (run any time)

In [ ]:
progress()

## Step 10 — Review: the with/without ablation

Once Step 9 reports complete, a real `3b` row is logged. This cell prints the
plain vs. CA comparison — the actual Phase 3b result.

In [ ]:
import pandas as pd
log = pd.read_csv('experiments/results/experiment_log.csv')
real = log[log['phase'].isin(['3a', '3b'])]
real = real[real['model'].isin(['yolov8n', 'yolov8n_ca'])]
cols = ['run_id','model','phase','map50','map50_95','precision','recall','f1',
        'param_count']
print(real[cols].to_string(index=False) if not real.empty else 'No 3a/3b yolo rows yet.')

## Final step — Save back to Drive

Then download `experiments/results/experiment_log.csv` and the
`experiments/runs/yolov8n_ca_source/` folder, drop them in your local repo's
`incoming/` folder, and tell your assistant — the config-hash check + merge
is the same as the YOLO26n handoff.

In [ ]:
out = save_results()
if out.is_dir():
    print("\nContents:")
    for p in sorted(out.rglob('*')):
        if p.is_file():
            print(" ", p.relative_to(out), f"{p.stat().st_size / 1e6:.1f} MB")